In [8]:
import os
import re
import time
import torch
import torch
import random
import bisect
import json
from pathlib import Path
from tokenizers import Tokenizer
from transformers import GPT2Model, GPT2LMHeadModel, GPT2Config, LlamaModel, LlamaForCausalLM, PreTrainedModel 
from samplings import top_p_sampling, top_k_sampling, temperature_sampling
from abctoolkit.utils import Exclaim_re, Quote_re, SquareBracket_re, Barline_regexPattern
from abctoolkit.transpose import Note_list, Pitch_sign_list
from abctoolkit.duration import calculate_bartext_duration

In [9]:
# Configurations for inference
# INFERENCE_WEIGHTS_PATH = '../weights/weights_notagenx_p_size_16_p_length_1024_p_layers_20_h_size_1280.pth'               # Path to weights for inference# Folder to save output files
INFERENCE_WEIGHTS_PATH = "/usr/xtmp/cy232/accompgen/weights/weights_notagen_10_epoch_p_size_16_p_length_1024_p_layers_20_c_layers_6_h_size_1280_lr_1e-05_batch_1.pth"
TOP_K = 9                                                       # Top k for sampling
TOP_P = 0.9                                                      # Top p for sampling
TEMPERATURE = 1.2                                                 # Temperature for sampling

# Configurations for model
PATCH_STREAM = True                                             # Stream training / inference
PATCH_SIZE = 16                                                # Patch Size
PATCH_LENGTH = 1024                                             # Patch Length
CHAR_NUM_LAYERS = 6                                             # Number of layers in the decoder
PATCH_NUM_LAYERS = 20                                           # Number of layers in the encoder
HIDDEN_SIZE = 1280                                               # Hidden Size
MOTIF_ATTENTION_BIAS = 2.0                                       # Scalar added to QK attention logits for motif patch key positions before softmax

# --- Backbone selection -------------------------------------------------------
# USE_STAGE1_STUDENT=True loads the Stage-2 distilled model: a wider llama patch
# encoder + projection feeding the original 1280-d gpt2 char decoder. False = the
# original gpt2 NotaGen. The checkpoint at INFERENCE_WEIGHTS_PATH must MATCH this
# toggle (a Stage-2 checkpoint carries llama-encoder + patch_proj keys).
ENCODER_BACKBONE = "llama"
DECODER_BACKBONE = "gpt2"
USE_STAGE1_STUDENT = True
STUDENT_ENCODER_BACKBONE = "llama"
STUDENT_PATCH_NUM_LAYERS = 24
STUDENT_HIDDEN_SIZE = 1536

device = torch.device("cuda")

In [10]:
class Patchilizer:
    def __init__(self, stream=PATCH_STREAM):
        self.stream = stream
        self.delimiters = ["|:", "::", ":|", "[|", "||", "|]", "|"]
        self.regexPattern = '(' + '|'.join(map(re.escape, self.delimiters)) + ')'
        self.bos_token_id = 1
        self.eos_token_id = 2
        self.special_token_id = 0

    def split_bars(self, body_lines):
        """
        Split a body of music into individual bars.
        """
        new_bars = []
        try:
            for line in body_lines:
                line_bars = re.split(self.regexPattern, line)
                line_bars = list(filter(None, line_bars))
                new_line_bars = []

                if len(line_bars) == 1:
                    new_line_bars = line_bars
                else:
                    if line_bars[0] in self.delimiters:
                        new_line_bars = [line_bars[i] + line_bars[i + 1] for i in range(0, len(line_bars), 2)]
                    else:
                        new_line_bars = [line_bars[0]] + [line_bars[i] + line_bars[i + 1] for i in range(1, len(line_bars), 2)]
                    if 'V' not in new_line_bars[-1]:
                        new_line_bars[-2] += new_line_bars[-1]  # 吸收最后一个 小节线+\n 的组合
                        new_line_bars = new_line_bars[:-1]
                new_bars += new_line_bars
        except:
            pass

        return new_bars

    def split_patches(self, abc_text, patch_size=PATCH_SIZE, generate_last=False):
        if not generate_last and len(abc_text) % patch_size != 0:
            abc_text += chr(self.eos_token_id)
        patches = [abc_text[i : i + patch_size] for i in range(0, len(abc_text), patch_size)]
        return patches

    def patch2chars(self, patch):
        """
        Convert a patch into a bar.
        """
        bytes = ''
        for idx in patch:
            if idx == self.eos_token_id:
                break
            if idx < self.eos_token_id:
                pass
            bytes += chr(idx)
        return bytes
        

    def patchilize_metadata(self, metadata_lines):

        metadata_patches = []
        for line in metadata_lines:
            metadata_patches += self.split_patches(line)

        return metadata_patches
    
    def patchilize_tunebody(self, tunebody_lines, encode_mode='train'):

        tunebody_patches = []
        bars = self.split_bars(tunebody_lines)
        if encode_mode == 'train':
            for bar in bars:
                tunebody_patches += self.split_patches(bar)
        elif encode_mode == 'generate':
            for bar in bars[:-1]:
                tunebody_patches += self.split_patches(bar)
            tunebody_patches += self.split_patches(bars[-1], generate_last=True)
       
        return tunebody_patches

    def encode_train(self, abc_text, patch_length=PATCH_LENGTH, patch_size=PATCH_SIZE, add_special_patches=True, cut=True):

        lines = abc_text.split('\n')
        lines = list(filter(None, lines))
        lines = [line + '\n' for line in lines]

        tunebody_index = -1
        for i, line in enumerate(lines):
            if '[V:' in line:
                tunebody_index = i
                break

        metadata_lines = lines[ : tunebody_index]
        tunebody_lines = lines[tunebody_index : ]

        if self.stream:
            tunebody_lines = ['[r:' + str(line_index) + '/' + str(len(tunebody_lines) - line_index - 1) + ']' + line for line_index, line in
                                enumerate(tunebody_lines)]    

        metadata_patches = self.patchilize_metadata(metadata_lines)
        tunebody_patches = self.patchilize_tunebody(tunebody_lines, encode_mode='train')

        if add_special_patches:
            bos_patch = chr(self.bos_token_id) * (patch_size - 1) + chr(self.eos_token_id)
            eos_patch = chr(self.bos_token_id) + chr(self.eos_token_id) * (patch_size - 1)

            metadata_patches = [bos_patch] + metadata_patches
            tunebody_patches = tunebody_patches + [eos_patch]

        if self.stream:
            if len(metadata_patches) + len(tunebody_patches) > patch_length:
                available_cut_indexes = [0] + [index + 1 for index, patch in enumerate(tunebody_patches) if '\n' in patch]
                line_index_for_cut_index = list(range(len(available_cut_indexes)))  
                end_index = len(metadata_patches) + len(tunebody_patches) - patch_length
                biggest_index = bisect.bisect_left(available_cut_indexes, end_index) 
                available_cut_indexes = available_cut_indexes[:biggest_index + 1]

                if len(available_cut_indexes) == 1:
                    choices = ['head']
                elif len(available_cut_indexes) == 2:
                    choices = ['head', 'tail']
                else:
                    choices = ['head', 'tail', 'middle']
                choice = random.choice(choices)
                if choice == 'head':
                    patches = metadata_patches + tunebody_patches[0:]
                else:
                    if choice == 'tail':
                        cut_index = len(available_cut_indexes) - 1
                    else:
                        cut_index = random.choice(range(1, len(available_cut_indexes) - 1))

                    line_index = line_index_for_cut_index[cut_index] 
                    stream_tunebody_lines = tunebody_lines[line_index : ]
                    
                    stream_tunebody_patches = self.patchilize_tunebody(stream_tunebody_lines, encode_mode='train')
                    if add_special_patches:
                        stream_tunebody_patches = stream_tunebody_patches + [eos_patch]
                    patches = metadata_patches + stream_tunebody_patches
            else:
                patches = metadata_patches + tunebody_patches
        else:
            patches = metadata_patches + tunebody_patches

        if cut: 
            patches = patches[ : patch_length]
        else:   
            pass

        # encode to ids
        id_patches = []
        for patch in patches:
            id_patch = [ord(c) for c in patch] + [self.special_token_id] * (patch_size - len(patch))
            id_patches.append(id_patch)

        return id_patches

    def encode_generate(self, abc_code, patch_length=PATCH_LENGTH, patch_size=PATCH_SIZE, add_special_patches=True):

        lines = abc_code.split('\n')
        lines = list(filter(None, lines))
    
        tunebody_index = None
        for i, line in enumerate(lines):
            if line.startswith('[V:') or line.startswith('[r:'):
                tunebody_index = i
                break
    
        metadata_lines = lines[ : tunebody_index]
        tunebody_lines = lines[tunebody_index : ]   
    
        metadata_lines = [line + '\n' for line in metadata_lines]
        if self.stream:
            if not abc_code.endswith('\n'):
                tunebody_lines = [tunebody_lines[i] + '\n' for i in range(len(tunebody_lines) - 1)] + [tunebody_lines[-1]]
            else:
                tunebody_lines = [tunebody_lines[i] + '\n' for i in range(len(tunebody_lines))]
        else:
            tunebody_lines = [line + '\n' for line in tunebody_lines]
    
        metadata_patches = self.patchilize_metadata(metadata_lines)
        tunebody_patches = self.patchilize_tunebody(tunebody_lines, encode_mode='generate')
    
        if add_special_patches:
            bos_patch = chr(self.bos_token_id) * (patch_size - 1) + chr(self.eos_token_id)

            metadata_patches = [bos_patch] + metadata_patches
    
        patches = metadata_patches + tunebody_patches
        patches = patches[ : patch_length]

        # encode to ids
        id_patches = []
        for patch in patches:
            if len(patch) < PATCH_SIZE and patch[-1] != chr(self.eos_token_id):
                id_patch = [ord(c) for c in patch]
            else:
                id_patch = [ord(c) for c in patch] + [self.special_token_id] * (patch_size - len(patch))
            id_patches.append(id_patch)
        
        return id_patches

    def decode(self, patches):
        """
        Decode patches into music.
        """
        return ''.join(self.patch2chars(patch) for patch in patches)


class PatchLevelDecoder(PreTrainedModel):
    """
    A Patch-level Decoder model for generating patch features in an auto-regressive manner.
    It inherits PreTrainedModel from transformers.
    """
    def __init__(self, config):
        super().__init__(config)
        self.patch_embedding = torch.nn.Linear(PATCH_SIZE * 128, config.n_embd)
        torch.nn.init.normal_(self.patch_embedding.weight, std=0.02)
        self.base = GPT2Model(config)
        self._motif_bias = None
        for block in self.base.h:
            block.register_forward_pre_hook(self._inject_motif_bias, with_kwargs=True)

    def _inject_motif_bias(self, module, args, kwargs):
        if self._motif_bias is not None:
            mask = kwargs.get('attention_mask')
            if mask is not None:
                kwargs['attention_mask'] = mask + self._motif_bias.to(mask.dtype)
        return args, kwargs

    def forward(self,
                patches: torch.Tensor,
                masks=None,
                motif_attention_bias=None) -> torch.Tensor:
        patches = torch.nn.functional.one_hot(patches, num_classes=128).to(self.dtype)
        patches = patches.reshape(len(patches), -1, PATCH_SIZE * (128))
        patches = self.patch_embedding(patches.to(self.device))

        self._motif_bias = motif_attention_bias[:, None, None, :] if motif_attention_bias is not None else None

        try:
            if masks is None:
                result = self.base(inputs_embeds=patches)
            else:
                result = self.base(inputs_embeds=patches, attention_mask=masks)
        finally:
            self._motif_bias = None

        return result


class CharLevelDecoder(PreTrainedModel):
    """
    A Char-level Decoder model for generating the chars within each patch in an auto-regressive manner
    based on the encoded patch features. It inherits PreTrainedModel from transformers.
    """
    def __init__(self, config):
        super().__init__(config)
        self.special_token_id = 0
        self.bos_token_id = 1

        self.base = GPT2LMHeadModel(config)

    def forward(self,
                encoded_patches: torch.Tensor,
                target_patches: torch.Tensor):
        """
        The forward pass of the char-level decoder model.
        :param encoded_patches: the encoded patches
        :param target_patches: the target patches
        :return: the output of the model
        """
        # preparing the labels for model training
        target_patches = torch.cat((torch.ones_like(target_patches[:,0:1])*self.bos_token_id, target_patches), dim=1)
        # print('target_patches shape:', target_patches.shape)

        target_masks = target_patches == self.special_token_id
        labels = target_patches.clone().masked_fill_(target_masks, -100)

        # masking the labels for model training
        target_masks = torch.ones_like(labels)
        target_masks = target_masks.masked_fill_(labels == -100, 0)

        # select patches
        if PATCH_SAMPLING_BATCH_SIZE!=0 and PATCH_SAMPLING_BATCH_SIZE<target_patches.shape[0]:
            indices = list(range(len(target_patches)))
            random.shuffle(indices)
            selected_indices = sorted(indices[:PATCH_SAMPLING_BATCH_SIZE])

            target_patches = target_patches[selected_indices,:]
            target_masks = target_masks[selected_indices,:]
            encoded_patches = encoded_patches[selected_indices,:]

        # get input embeddings
        inputs_embeds = torch.nn.functional.embedding(target_patches, self.base.transformer.wte.weight)

        # concatenate the encoded patches with the input embeddings
        inputs_embeds = torch.cat((encoded_patches.unsqueeze(1), inputs_embeds[:,1:,:]), dim=1)

        output = self.base(inputs_embeds=inputs_embeds, 
                         attention_mask=target_masks,
                         labels=labels)
                         # output_hidden_states=True=True)

        return output

    def generate(self,
                 encoded_patch: torch.Tensor,   # [hidden_size]
                 tokens: torch.Tensor): # [1]
        """
        The generate function for generating a patch based on the encoded patch and already generated tokens.
        :param encoded_patch: the encoded patch
        :param tokens: already generated tokens in the patch
        :return: the probability distribution of next token
        """
        encoded_patch = encoded_patch.reshape(1, 1, -1) # [1, 1, hidden_size]
        tokens = tokens.reshape(1, -1)

        # Get input embeddings
        tokens = torch.nn.functional.embedding(tokens, self.base.transformer.wte.weight)

        # Concatenate the encoded patch with the input embeddings
        tokens = torch.cat((encoded_patch, tokens[:,1:,:]), dim=1)
        
        # Get output from model
        outputs = self.base(inputs_embeds=tokens)
        
        # Get probabilities of next token
        probs = torch.nn.functional.softmax(outputs.logits.squeeze(0)[-1], dim=-1)

        return probs

class NotaGenLMHeadModel(PreTrainedModel):
    """
    NotaGen is a language model with a hierarchical structure.
    It includes a patch-level decoder and a char-level decoder.
    The patch-level decoder is used to generate patch features in an auto-regressive manner.
    The char-level decoder is used to generate the chars within each patch in an auto-regressive manner.
    It inherits PreTrainedModel from transformers.
    """
    def __init__(self, encoder_config, decoder_config):
        super().__init__(encoder_config)
        self.special_token_id = 0
        self.bos_token_id = 1
        self.eos_token_id = 2
        self.patch_level_decoder = PatchLevelDecoder(encoder_config)
        self.char_level_decoder = CharLevelDecoder(decoder_config)

    def forward(self,
                patches: torch.Tensor,
                masks: torch.Tensor,
                motif_weights: torch.Tensor = None):
        patches = patches.reshape(len(patches), -1, PATCH_SIZE)

        motif_attention_bias = None
        if motif_weights is not None:
            motif_attention_bias = (motif_weights > 1.0).float() * MOTIF_ATTENTION_BIAS * 1000

        encoded_patches = self.patch_level_decoder(patches, masks, motif_attention_bias)["last_hidden_state"]
        
        left_shift_masks = masks * (masks.flip(1).cumsum(1).flip(1) > 1)
        masks[:, 0] = 0
        
        encoded_patches = encoded_patches[left_shift_masks == 1]
        patches = patches[masks == 1]        

        return self.char_level_decoder(encoded_patches, patches)
        
    def generate(self,
                 patches: torch.Tensor,
                 top_k=0,
                 top_p=1,
                 temperature=1.0):
        """
        The generate function for generating patches based on patches.
        :param patches: the patches to be encoded
        :param top_k: the top k for sampling
        :param top_p: the top p for sampling
        :param temperature: the temperature for sampling
        :return: the generated patches
        """
        if patches.shape[-1] % PATCH_SIZE != 0:
            tokens = patches[:,:,-(patches.shape[-1]%PATCH_SIZE):].squeeze(0, 1)
            tokens = torch.cat((torch.tensor([self.bos_token_id], device=self.device), tokens), dim=-1)
            patches = patches[:,:,:-(patches.shape[-1]%PATCH_SIZE)]
        else:
            tokens =  torch.tensor([self.bos_token_id], device=self.device)

        patches = patches.reshape(len(patches), -1, PATCH_SIZE) # [bs, seq, patch_size]
        encoded_patches = self.patch_level_decoder(patches)["last_hidden_state"]    # [bs, seq, hidden_size]
        generated_patch = []            

        while True:
            prob = self.char_level_decoder.generate(encoded_patches[0][-1], tokens).cpu().detach().numpy()  # [128]
            prob = top_k_sampling(prob, top_k=top_k, return_probs=True) # [128]
            prob = top_p_sampling(prob, top_p=top_p, return_probs=True) # [128]
            token = temperature_sampling(prob, temperature=temperature) # int
            char = chr(token)
            generated_patch.append(token)

            if len(tokens) >= PATCH_SIZE:# or token == self.eos_token_id:
                break
            else:
                tokens = torch.cat((tokens, torch.tensor([token], device=self.device)), dim=0)
        
        return generated_patch

def clean_to_abc(raw_text, unreduce=True, output_path='output.abc'):
    # Remove [r:x/y] tags
    cleaned = re.sub(r'\[r:\d+/\d+\]', '', raw_text)

    # Add required ABC headers
    lines = cleaned.strip().splitlines()
    header_inserted = False
    abc_lines = []
    for line in lines:
        if not header_inserted and line.startswith('%%score'):
            abc_lines.insert(0, 'T:Generated\n')
            abc_lines.insert(0, 'X:1\n')
            header_inserted = True
        abc_lines.append(line if line.endswith('\n') else line + '\n')

    # Optional: fill missing rests
    if unreduce:
        try:
            abc_lines = rest_unreduce(abc_lines)
        except Exception as e:
            print("Unreduce failed:", e)

    # Save to .abc file
    Path(output_path).write_text(''.join(abc_lines), encoding='utf-8')
    print(f"Saved cleaned ABC to {output_path}")
    return output_path

# --- Use the centralized, backbone-agnostic model (supports llama encoder + patch_proj) ---
import os as _os, sys as _sys
_sys.path.insert(0, _os.path.abspath(_os.path.join(_os.getcwd(), "..")))
from notagen_core import PatchLevelDecoder, CharLevelDecoder, NotaGenLMHeadModel, safe_normalize_probs

In [11]:
Note_list = Note_list + ['z', 'x']

patchilizer = Patchilizer()

from notagen_core import build_notagen_configs
if USE_STAGE1_STUDENT:
    patch_config, byte_config = build_notagen_configs(
        encoder_backbone=STUDENT_ENCODER_BACKBONE, decoder_backbone=DECODER_BACKBONE,
        patch_num_layers=STUDENT_PATCH_NUM_LAYERS, char_num_layers=CHAR_NUM_LAYERS,
        hidden_size=HIDDEN_SIZE, encoder_hidden_size=STUDENT_HIDDEN_SIZE,
        patch_length=PATCH_LENGTH, patch_size=PATCH_SIZE,
        motif_attention_bias=MOTIF_ATTENTION_BIAS, patch_sampling_batch_size=0)
else:
    patch_config, byte_config = build_notagen_configs(
        encoder_backbone=ENCODER_BACKBONE, decoder_backbone=DECODER_BACKBONE,
        patch_num_layers=PATCH_NUM_LAYERS, char_num_layers=CHAR_NUM_LAYERS,
        hidden_size=HIDDEN_SIZE, patch_length=PATCH_LENGTH, patch_size=PATCH_SIZE,
        motif_attention_bias=MOTIF_ATTENTION_BIAS, patch_sampling_batch_size=0)

model = NotaGenLMHeadModel(encoder_config=patch_config, decoder_config=byte_config).to(device)

def prepare_model_for_kbit_training(model, use_gradient_checkpointing=True):
    """
    Prepare model for k-bit training.
    Features include:
    1. Convert model to mixed precision (FP16).
    2. Disable unnecessary gradient computations.
    3. Enable gradient checkpointing (optional).
    """
    # Convert model to mixed precision
    model = model.to(dtype=torch.float16)

    # Disable gradients for embedding layers
    for param in model.parameters():
        if param.dtype == torch.float32:
            param.requires_grad = False

    # Enable gradient checkpointing
    if use_gradient_checkpointing:
        model.gradient_checkpointing_enable()

    return model


model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=False  
)

print("Parameter Number: " + str(sum(p.numel() for p in model.parameters() if p.requires_grad)))

checkpoint = torch.load(INFERENCE_WEIGHTS_PATH, map_location=torch.device(device))
model.load_state_dict(checkpoint['model'])
model = model.to(device)
model.eval()

def complete_brackets(s):
    stack = []
    bracket_map = {'{': '}', '[': ']', '(': ')'}
    
    # Iterate through each character, handle bracket matching
    for char in s:
        if char in bracket_map:
            stack.append(char)
        elif char in bracket_map.values():
            # Find the corresponding left bracket
            for key, value in bracket_map.items():
                if value == char:
                    if stack and stack[-1] == key:
                        stack.pop()
                    break  # Found matching right bracket, process next character
    
    # Complete missing right brackets (in reverse order of remaining left brackets in stack)
    completion = ''.join(bracket_map[c] for c in reversed(stack))
    return s + completion


def rest_unreduce(abc_lines):

    tunebody_index = None
    for i in range(len(abc_lines)):
        if abc_lines[i].startswith('%%score'):
            abc_lines[i] = complete_brackets(abc_lines[i])
        if '[V:' in abc_lines[i]:
            tunebody_index = i
            break

    metadata_lines = abc_lines[: tunebody_index]
    tunebody_lines = abc_lines[tunebody_index:]

    part_symbol_list = []
    voice_group_list = []
    for line in metadata_lines:
        if line.startswith('%%score'):
            for round_bracket_match in re.findall(r'\((.*?)\)', line):
                voice_group_list.append(round_bracket_match.split())
            existed_voices = [item for sublist in voice_group_list for item in sublist]
        if line.startswith('V:'):
            symbol = line.split()[0]
            part_symbol_list.append(symbol)
            if symbol[2:] not in existed_voices:
                voice_group_list.append([symbol[2:]])
    z_symbol_list = []  # voices that use z as rest
    x_symbol_list = []  # voices that use x as rest
    for voice_group in voice_group_list:
        z_symbol_list.append('V:' + voice_group[0])
        for j in range(1, len(voice_group)):
            x_symbol_list.append('V:' + voice_group[j])

    part_symbol_list.sort(key=lambda x: int(x[2:]))

    unreduced_tunebody_lines = []

    for i, line in enumerate(tunebody_lines):
        unreduced_line = ''

        line = re.sub(r'^\[r:[^\]]*\]', '', line)

        pattern = r'\[V:(\d+)\](.*?)(?=\[V:|$)'
        matches = re.findall(pattern, line)

        line_bar_dict = {}
        for match in matches:
            key = f'V:{match[0]}'
            value = match[1]
            line_bar_dict[key] = value

        # calculate duration and collect barline
        dur_dict = {}  
        for symbol, bartext in line_bar_dict.items():
            right_barline = ''.join(re.split(Barline_regexPattern, bartext)[-2:])
            bartext = bartext[:-len(right_barline)]
            try:
                bar_dur = calculate_bartext_duration(bartext)
            except:
                bar_dur = None
            if bar_dur is not None:
                if bar_dur not in dur_dict.keys():
                    dur_dict[bar_dur] = 1
                else:
                    dur_dict[bar_dur] += 1

        try:
            ref_dur = max(dur_dict, key=dur_dict.get)
        except:
            pass    # use last ref_dur

        if i == 0:
            prefix_left_barline = line.split('[V:')[0]
        else:
            prefix_left_barline = ''

        for symbol in part_symbol_list:
            if symbol in line_bar_dict.keys():
                symbol_bartext = line_bar_dict[symbol]
            else:
                if symbol in z_symbol_list:
                    symbol_bartext = prefix_left_barline + 'z' + str(ref_dur) + right_barline
                elif symbol in x_symbol_list:
                    symbol_bartext = prefix_left_barline + 'x' + str(ref_dur) + right_barline
            unreduced_line += '[' + symbol + ']' + symbol_bartext

        unreduced_tunebody_lines.append(unreduced_line + '\n')

    unreduced_lines = metadata_lines + unreduced_tunebody_lines

    return unreduced_lines


def inference_patch(period, composer, instrumentation, additional_prompts=[], tune_body = []):

    prompt_lines=[
    '%' + period + '\n',
    '%' + composer + '\n',
    '%' + instrumentation + '\n']

    for prompt in additional_prompts:
        prompt_lines.append(prompt)

    for prompt in tune_body:
        prompt_lines.append(prompt)

    tunebody_flag = tune_body != []

    while True:

        failure_flag = False

        bos_patch = [patchilizer.bos_token_id] * (PATCH_SIZE - 1) + [patchilizer.eos_token_id]

        start_time = time.time()

        prompt_patches = patchilizer.patchilize_metadata(prompt_lines)
        byte_list = list(''.join(prompt_lines))
        context_tunebody_byte_list = []
        metadata_byte_list = []

        print(''.join(byte_list), end='')

        prompt_patches = [[ord(c) for c in patch] + [patchilizer.special_token_id] * (PATCH_SIZE - len(patch)) for patch
                          in prompt_patches]
        prompt_patches.insert(0, bos_patch)

        input_patches = torch.tensor(prompt_patches, device=device).reshape(1, -1)

        end_flag = False
        cut_index = None

        tunebody_flag = tunebody_flag == True

        with torch.inference_mode():
            
            while True:
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    predicted_patch = model.generate(input_patches.unsqueeze(0),
                                                    top_k=TOP_K,
                                                    top_p=TOP_P,
                                                    temperature=TEMPERATURE)
                if not tunebody_flag and patchilizer.decode([predicted_patch]).startswith('[r:'):  # 初次进入tunebody，必须以[r:0/开头
                    tunebody_flag = True
                    r0_patch = torch.tensor([ord(c) for c in '[r:0/']).unsqueeze(0).to(device)
                    temp_input_patches = torch.concat([input_patches, r0_patch], axis=-1)
                    predicted_patch = model.generate(temp_input_patches.unsqueeze(0),
                                                    top_k=TOP_K,
                                                    top_p=TOP_P,
                                                    temperature=TEMPERATURE)
                    predicted_patch = [ord(c) for c in '[r:0/'] + predicted_patch
                if predicted_patch[0] == patchilizer.bos_token_id and predicted_patch[1] == patchilizer.eos_token_id:
                    end_flag = True
                    break
                next_patch = patchilizer.decode([predicted_patch])

                for char in next_patch:
                    byte_list.append(char)
                    if tunebody_flag:
                        context_tunebody_byte_list.append(char)
                    else:
                        metadata_byte_list.append(char)
                    print(char, end='')

                patch_end_flag = False
                for j in range(len(predicted_patch)):
                    if patch_end_flag:
                        predicted_patch[j] = patchilizer.special_token_id
                    if predicted_patch[j] == patchilizer.eos_token_id:
                        patch_end_flag = True

                predicted_patch = torch.tensor([predicted_patch], device=device)  # (1, 16)
                input_patches = torch.cat([input_patches, predicted_patch], dim=1)  # (1, 16 * patch_len)

                if len(byte_list) > 102400:
                    failure_flag = True
                    break
                if time.time() - start_time > 10 * 60: 
                    failure_flag = True
                    break

                if input_patches.shape[1] >= PATCH_LENGTH * PATCH_SIZE and not end_flag:
                    print('Stream generating...')

                    metadata = ''.join(metadata_byte_list)
                    context_tunebody = ''.join(context_tunebody_byte_list)

                    if '\n' not in context_tunebody:
                        break   # Generated content is all metadata, abandon

                    context_tunebody_liness = context_tunebody.split('\n')
                    if not context_tunebody.endswith('\n'):
                        context_tunebody_liness = [context_tunebody_liness[i] + '\n' for i in range(len(context_tunebody_liness) - 1)] + [context_tunebody_liness[-1]]
                    else:
                        context_tunebody_liness = [context_tunebody_liness[i] + '\n' for i in range(len(context_tunebody_liness))]

                    cut_index = len(context_tunebody_liness) // 2
                    abc_code_slice = metadata + ''.join(context_tunebody_liness[-cut_index:])

                    input_patches = patchilizer.encode_generate(abc_code_slice)

                    input_patches = [item for sublist in input_patches for item in sublist]
                    input_patches = torch.tensor([input_patches], device=device)
                    input_patches = input_patches.reshape(1, -1)

                    context_tunebody_byte_list = list(''.join(context_tunebody_lines[-cut_index:]))

            if not failure_flag:
                abc_text = ''.join(byte_list)

                # unreduce
                abc_lines = abc_text.split('\n')
                abc_lines = list(filter(None, abc_lines))
                abc_lines = [line + '\n' for line in abc_lines]
                try:
                    unreduced_abc_lines = rest_unreduce(abc_lines)
                except:
                    failure_flag = True
                    pass
                else:
                    unreduced_abc_lines = [line for line in unreduced_abc_lines if not(line.startswith('%') and not line.startswith('%%'))]
                    unreduced_abc_lines = ['X:1\n'] + unreduced_abc_lines
                    unreduced_abc_text = ''.join(unreduced_abc_lines)
                    return unreduced_abc_text

Parameter Number: 1029413888


In [12]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version built:", torch.version.cuda)

torch: 2.3.0
cuda available: True
cuda version built: 11.8


In [13]:
# === Motif helpers (realization + attention inspection) ===
# Full source in notebook/motif_helpers.py. The realizer turns an abstract
# `%motif:v1:` line into a `%motif:v1:` + `%motif:abc:` pair, sourcing the
# realization from training data when possible, else generating it algorithmically.
from motif_helpers import realize_motif_line, inspect_motif_attention, compare_bias_on_off

In [23]:
period = "Romantic"
composer = "Schubert, Franz"
instrumentation = "Art Song"

def sanitize(name: str) -> str:
    """Convert to safe filename (remove spaces, commas, etc.)."""
    return re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_')


print(f"Period: {period}, Composer: {composer}, Instrumentation: {instrumentation}")
            

try:
    period, composer, instrumentation = 'Romantic', 'Schubert, Franz', 'Art Song'
    default_note, tempo, meter, key = "1/8", "1/4=60", "3/4", "C"
    # score = "1 { ( 2 4 ) | ( 3 5 ) }"

    motif = "%motif:v1:step_skip_leap: 0,3,-1,-1 ; 0,-2,1,1 ; 0,-1,-1,-1\n"
    # motif = "%motif:v1:step_skip_leap: 0,2,2,2 ; 0, 3,3,3 ; 0,-3,-3,-3\n"
    # --- optional: pre-realize the motif so the model gets a concrete %motif:abc line ---
    USE_REALIZED_MOTIF = True   # set False to fall back to the bare %motif:v1: prompt
    REALIZATION_KEY    = key    # which key to source realizations from
    if USE_REALIZED_MOTIF:
        motif, _motif_sources = realize_motif_line(motif, key=REALIZATION_KEY)
        print('Sending motif prompt:\n' + motif)
    
    # motif = "%motif:v1:step_skip_leap: 0,-1,-1,-1 (A2 | G2 F2 E2) ; 0,-1,3,-1 (F2 E2 A2 | (G2) ; 0,1,-1,-2 (e2 f2 | (ec) \n"

    # score = " 1 { 2 | ( 3 4 5 ) }"
    # voices = [motif, """V:1 treble nm="Singstimme"\n""",
    #             """V:2 treble nm="Pianoforte"\n""",
    #             """V:3 bass\n""",
    #             """V:4 bass\n""",
    #             """V:5 bass\n"""]
    # first_tune = """[r:0/80]"""
    # first_tune = """[r:0/80][V:1]"^Andante con moto"!p! g4{a} (gf/e/)|[V:3](C/G/E/G/) (C/G/E/G/) (C/G/E/G/)|\n"""
    # result = inference_patch(period, composer, instrumentation, [f"%%score {score} \n", f"L:{default_note}\n", f"Q:{tempo}\n", f"M:{meter}\n", f"K:{key}\n"]+ voices, first_tune)

    result = inference_patch(period, 
                            composer, 
                            instrumentation, 
                            [motif]
                            )

    abc_lines = result.splitlines()
    abc_lines = [line + '\n' for line in abc_lines if line.strip()]  # Add newlines and remove empty lines

    abc_lines = rest_unreduce(abc_lines)

    # Construct unique file name
    filename = f"{sanitize(period)}_{sanitize(composer)}_{sanitize(instrumentation)}.abc"
    
    with open(filename, "w", encoding="utf-8") as f:
        f.writelines(abc_lines)
    !python abc2xml.py -o . {filename}
except Exception as e:
    print(f"Some error {e} has occured, skipping{filename}")

Period: Romantic, Composer: Schubert, Franz, Instrumentation: Art Song
Realized motif prompt for key=C:
  (0, 3, -1, -1) -> 'A,/ A,/ ^G,/ (A,/ A) ^G | G G z/ C/ D/ E/'  [training[C]]
  (0, -2, 1, 1) -> 'a2 f2) d2 | e4 f2'  [training[C]]
  (0, -1, -1, -1) -> 'e d c B c d'  [training[C]]
Sending motif prompt:
%motif:v1:step_skip_leap: 0,3,-1,-1 ; 0,-2,1,1 ; 0,-1,-1,-1
%motif:abc: A,/ A,/ ^G,/ (A,/ A) ^G | G G z/ C/ D/ E/ ; a2 f2) d2 | e4 f2 ; e d c B c d

%Romantic
%Schubert, Franz
%Art Song
%motif:v1:step_skip_leap: 0,3,-1,-1 ; 0,-2,1,1 ; 0,-1,-1,-1
%motif:abc: A,/ A,/ ^G,/ (A,/ A) ^G | G G z/ C/ D/ E/ ; a2 f2) d2 | e4 f2 ; e d c B c d
%%score 1 { 2 | ( 3 4 ) }
L:1/8
Q:1/4=100
M:2/4
K:G
V:1 treble nm="Voice"
V:2 treble nm="Piano"
V:3 bass 
V:4 bass 
[r:0/54][V:1]"^Lebhaft" z4|[V:2]!p! (B,/D/G/B/ d/B/G/D/)|[V:3][G,,D,G,]4|
[r:1/53][V:2](B,/D/G/B/ d/B/G/D/)|[V:3][G,,D,G,]4|
[r:2/52][V:2](C/D/F/D/ A/D/d/D/)|[V:3](A,2 C2)|[V:4][G,,D,]4|
[r:3/51][V:2](B,/D/G/B/ d/B/G/D/)|[V:3]G,4|[V:4][G,,D,

In [15]:
import difflib
import re
from pathlib import Path

TRAIN_ABC_DIR = Path('../data/abcfiles_processed')
CORPUS_JSON   = Path('../data/Lieder/data/corpus_conversion.json')

def _build_corpus_lookup():
    if not CORPUS_JSON.exists():
        return {}
    with open(CORPUS_JSON, encoding='utf-8') as f:
        entries = json.load(f)
    lookup = {}
    for entry in entries:
        path = Path(entry['in'])
        lc_id = path.stem
        parts = path.parts
        def _clean(s): return s.replace('_', ' ')
        composer   = _clean(parts[2]) if len(parts) > 2 else '?'
        collection = _clean(parts[3]) if len(parts) > 3 else '?'
        song       = _clean(parts[4]) if len(parts) > 4 else '?'
        lookup[lc_id] = (composer, collection, song)
    return lookup

_CORPUS_LOOKUP = _build_corpus_lookup()

def corpus_info(abc_filename):
    """Return (composer, collection, song) for an abc filename like lc4904021_C.abc."""
    stem = Path(abc_filename).stem
    lc_id = stem.split('_')[0]
    return _CORPUS_LOOKUP.get(lc_id, (None, None, None))

def extract_meter(abc_text):
    """Parse M: header -> (meter_type, beat_count).
    meter_type : 'simple' or 'compound'
    beat_count : beats per measure (2, 3, 4, ...)
    Compound: numerator % 3 == 0 and numerator > 3 and denominator >= 8
      e.g. 6/8->compound/2, 9/8->compound/3, 12/8->compound/4
    Returns (None, None) if no M: header found.
    """
    for line in abc_text.splitlines():
        m = re.match(r'^M:\s*(\d+)/(\d+)', line.strip())
        if m:
            num, den = int(m.group(1)), int(m.group(2))
            if num % 3 == 0 and num > 3 and den >= 8:
                return ('compound', num // 3)
            else:
                return ('simple', num)
    return (None, None)

def uninterleave_abc(abc_text):
    """Split interleaved ABC body lines (inline [V:N] markers) into per-voice
    note sequences. Returns (voice_order, voices_dict)."""
    voices = {}
    voice_order = []
    for line in abc_text.splitlines():
        s = line.strip()
        if not s:
            continue
        if re.match(r'^[A-WYZ]:', s) or s.startswith('%%') or s.startswith('%'):
            continue
        parts = re.split(r'(\[V:[^\]]+\])', s)
        current_voice = None
        for part in parts:
            vm = re.match(r'^\[V:([^\]]+)\]$', part)
            if vm:
                vid = vm.group(1).strip()
                if vid not in voices:
                    voices[vid] = []
                    voice_order.append(vid)
                current_voice = vid
            elif current_voice is not None and part.strip():
                voices[current_voice].append(part.strip())
    return voice_order, voices

def extract_note_content(abc_text):
    """Un-interleave voices and return notes in consistent (numerically sorted)
    voice order. Falls back to joining all non-header lines if no [V:] markers."""
    voice_order, voices = uninterleave_abc(abc_text)
    if voices:
        def _voice_key(v):
            try:    return (0, int(v))
            except: return (1, v)
        sorted_vids = sorted(voice_order, key=_voice_key)
        return ' '.join(' '.join(voices[v]) for v in sorted_vids)
    note_lines = []
    for line in abc_text.splitlines():
        s = line.strip()
        if not s or re.match(r'^[A-WYZ]:', s) or s.startswith('%%') or s.startswith('%'):
            continue
        note_lines.append(s)
    return ' '.join(note_lines)

def check_similarity(generated_filename, top_k=10):
    """Compare a generated ABC file against training data in abcfiles_processed.
    1. Parse generated meter (simple/compound + beat count).
    2. Skip training files with non-matching meter.
    3. Un-interleave both files before note-content comparison.
    """
    with open(generated_filename, 'r', encoding='utf-8') as f:
        gen_text = f.read()

    gen_meter_type, gen_beat_count = extract_meter(gen_text)
    gen_notes = extract_note_content(gen_text)

    train_files = list(TRAIN_ABC_DIR.rglob('*.abc'))
    print(f"Scanning {len(train_files)} training files across all keys...")
    print(f"Generated meter: {gen_meter_type}, {gen_beat_count} beats — filtering by matching meter first.")

    candidates = []
    skipped_meter = 0
    for train_file in train_files:
        try:
            train_text = train_file.read_text(encoding='utf-8')
            t_type, t_beats = extract_meter(train_text)
            if gen_meter_type is not None and t_type is not None:
                if t_type != gen_meter_type or t_beats != gen_beat_count:
                    skipped_meter += 1
                    continue
            train_notes = extract_note_content(train_text)
            quick = difflib.SequenceMatcher(None, gen_notes, train_notes, autojunk=False).quick_ratio()
            candidates.append((quick, train_file, train_notes))
        except Exception:
            continue

    print(f"Skipped {skipped_meter} non-matching-meter files. Comparing {len(candidates)} candidates.")
    candidates.sort(reverse=True)

    top_candidates = candidates[:max(top_k * 5, 50)]
    results = []
    for _, train_file, train_notes in top_candidates:
        exact = difflib.SequenceMatcher(None, gen_notes, train_notes, autojunk=False).ratio()
        results.append((exact, train_file))
    results.sort(reverse=True)

    print(f"\nTop {top_k} most similar training files to '{generated_filename}':")
    print(f"{'Sim':>6}  {'File':<28}  Composer — Collection — Song")
    print('-' * 100)
    for ratio, fpath in results[:top_k]:
        composer, collection, song = corpus_info(fpath.name)
        meta = f"{composer} — {collection} — {song}" if composer else "(not found in corpus)"
        print(f"{ratio:>6.4f}  {fpath.name:<28}  {meta}")

    top_score = results[0][0] if results else 0.0
    if top_score > 0.9:
        print("\n WARNING: Very high similarity ({:.0%}) — possible memorization/overfitting!".format(top_score))
    elif top_score > 0.7:
        print("\n High similarity ({:.0%}) — review top matches.".format(top_score))
    else:
        print(f"\nMax similarity: {top_score:.4f} — output appears novel.")

check_similarity(filename)


Scanning 21295 training files across all keys...
Generated meter: simple, 4 beats — filtering by matching meter first.
Skipped 13815 non-matching-meter files. Comparing 7480 candidates.

Top 10 most similar training files to 'Romantic_Schubert_Franz_Art_Song.abc':
   Sim  File                          Composer — Collection — Song
----------------------------------------------------------------------------------------------------
0.2180  lc6660019_A.abc               Netzel, Laura — 3 Lieder, Op.44 — 1 Lied
0.2124  lc6753259_Ab.abc              Burleigh, Harry Thacker —   — Nobody Knows de Trouble I’ve Seen
0.2069  lc6660019_Ab.abc              Netzel, Laura — 3 Lieder, Op.44 — 1 Lied
0.2035  lc6588799_Cb.abc              Browne, Augusta —   — The Reply of the Messenger Bird
0.2035  lc6588799_C.abc               Browne, Augusta —   — The Reply of the Messenger Bird
0.1960  lc6613436_Eb.abc              Grandval, Clémence de — 6 Nouvelles mélodies — 2 Si tu m’aimais!
0.1929  lc6753259_F.

In [16]:
import difflib
import re
from pathlib import Path
from abctoolkit.transpose import Note2pitch

TRAIN_ABC_DIR = Path('../data/abcfiles_processed')
CORPUS_JSON   = Path('../data/Lieder/data/corpus_conversion.json')

if '_CORPUS_LOOKUP' not in dir():
    def _build_corpus_lookup():
        if not CORPUS_JSON.exists():
            return {}
        with open(CORPUS_JSON, encoding='utf-8') as f:
            entries = json.load(f)
        lookup = {}
        for entry in entries:
            path = Path(entry['in'])
            lc_id = path.stem
            parts = path.parts
            def _clean(s): return s.replace('_', ' ')
            composer   = _clean(parts[2]) if len(parts) > 2 else '?'
            collection = _clean(parts[3]) if len(parts) > 3 else '?'
            song       = _clean(parts[4]) if len(parts) > 4 else '?'
            lookup[lc_id] = (composer, collection, song)
        return lookup
    _CORPUS_LOOKUP = _build_corpus_lookup()

def corpus_info(abc_filename):
    stem  = Path(abc_filename).stem
    lc_id = stem.split('_')[0]
    return _CORPUS_LOOKUP.get(lc_id, (None, None, None))

_NOTE_RE = re.compile(r"([_=^]{0,2})([A-Ga-g])([',]*)")

_SHARP_NOTES = ['F', 'C', 'G', 'D', 'A', 'E', 'B']
_FLAT_NOTES  = ['B', 'E', 'A', 'D', 'G', 'C', 'F']
_KEY_SIG_MAP = {
    'C': 0,  'G': 1,  'D': 2,  'A': 3,  'E': 4,  'B': 5,  'F#': 6, 'C#': 7,
    'F': -1, 'Bb': -2,'Eb': -3,'Ab': -4,'Db': -5,'Gb': -6,'Cb': -7,
    'Am': 0, 'Em': 1, 'Bm': 2, 'F#m': 3,'C#m': 4,'G#m': 5,'D#m': 6,'A#m': 7,
    'Dm': -1,'Gm': -2,'Cm': -3,'Fm': -4,'Bbm': -5,'Ebm': -6,'Abm': -7,
}

def _parse_key_sig(abc_text):
    """Return dict mapping note letter (uppercase) -> '^' or '_' per key signature."""
    m = re.search(r'^K:\s*(\S+)', abc_text, re.MULTILINE)
    if not m:
        return {}
    ks = m.group(1).strip()
    ks = re.sub(r'(maj(?:or)?)$', '', ks, flags=re.IGNORECASE)
    ks = re.sub(r'min(?:or)?$', 'm', ks, flags=re.IGNORECASE)
    if ks and ks[0].islower():
        ks = ks[0].upper() + ks[1:]
    n = _KEY_SIG_MAP.get(ks, 0)
    result = {}
    if n > 0:
        for note in _SHARP_NOTES[:n]:
            result[note] = '^'
    elif n < 0:
        for note in _FLAT_NOTES[:-n]:
            result[note] = '_'
    return result

def abc_to_intervals(abc_text):
    """Extract transposition-invariant interval sequence from ABC text.
    Un-interleaves voices first (consistent numerical order), then converts
    each note to MIDI pitch and returns semitone differences.
    """
    key_acc = _parse_key_sig(abc_text)
    pitches = []
    body_lines = []
    for line in abc_text.splitlines():
        s = line.strip()
        if not s or re.match(r'^[A-WYZ]:', s) or s.startswith('%%') or s.startswith('%'):
            continue
        body_lines.append(s)

    has_voice_markers = any('[V:' in ln for ln in body_lines)
    if has_voice_markers:
        voices = {}
        voice_order = []
        for ln in body_lines:
            parts = re.split(r'(\[V:[^\]]+\])', ln)
            current_voice = None
            for part in parts:
                vm = re.match(r'^\[V:([^\]]+)\]$', part)
                if vm:
                    vid = vm.group(1).strip()
                    if vid not in voices:
                        voices[vid] = []
                        voice_order.append(vid)
                    current_voice = vid
                elif current_voice is not None and part.strip():
                    voices[current_voice].append(part.strip())
        def _voice_key(v):
            try:    return (0, int(v))
            except: return (1, v)
        sorted_vids = sorted(voice_order, key=_voice_key)
        content = ' '.join(' '.join(voices[v]) for v in sorted_vids)
    else:
        content = ' '.join(body_lines)

    for acc, note, oct_mod in _NOTE_RE.findall(content):
        note_upper = note.upper()
        # Apply key signature when no explicit accidental
        effective_acc = acc if acc else key_acc.get(note_upper, '')
        for lookup_key in (effective_acc + note, effective_acc + note_upper):
            if lookup_key in Note2pitch:
                pitch = Note2pitch[lookup_key]
                for ch in oct_mod:
                    if ch == "'":  pitch += 12
                    elif ch == ',': pitch -= 12
                pitches.append(pitch)
                break

    if len(pitches) < 2:
        return []
    return [pitches[i+1] - pitches[i] for i in range(len(pitches) - 1)]
def check_similarity_intervals(generated_filename, top_k=10):
    """Interval-based similarity check (transposition-invariant).
    Un-interleaves voices and filters by meter before comparing.
    """
    gen_text = open(generated_filename, encoding='utf-8').read()
    gen_meter_type, gen_beat_count = extract_meter(gen_text)
    gen_intervals = abc_to_intervals(gen_text)
    if not gen_intervals:
        print("Could not extract any notes from the generated file.")
        return
    gen_seq = [str(i) for i in gen_intervals]

    train_files = list(TRAIN_ABC_DIR.rglob('*.abc'))
    print(f"Scanning {len(train_files)} training files (interval-based, all keys)...")
    print(f"Generated meter: {gen_meter_type}, {gen_beat_count} beats — filtering by matching meter first.")

    candidates = []
    skipped_meter = 0
    for train_file in train_files:
        try:
            train_text = train_file.read_text(encoding='utf-8')
            t_type, t_beats = extract_meter(train_text)
            if gen_meter_type is not None and t_type is not None:
                if t_type != gen_meter_type or t_beats != gen_beat_count:
                    skipped_meter += 1
                    continue
            train_intervals = abc_to_intervals(train_text)
            if not train_intervals:
                continue
            train_seq = [str(i) for i in train_intervals]
            quick = difflib.SequenceMatcher(None, gen_seq, train_seq, autojunk=False).quick_ratio()
            candidates.append((quick, train_file, train_seq))
        except Exception:
            continue

    print(f"Skipped {skipped_meter} non-matching-meter files. Comparing {len(candidates)} candidates.")
    candidates.sort(reverse=True)

    top_candidates = candidates[:max(top_k * 5, 50)]
    results = []
    for _, train_file, train_seq in top_candidates:
        exact = difflib.SequenceMatcher(None, gen_seq, train_seq, autojunk=False).ratio()
        results.append((exact, train_file))
    results.sort(reverse=True)

    print(f"\nTop {top_k} most similar training files to '{generated_filename}' (interval-based):")
    print(f"{'Sim':>6}  {'File':<28}  Composer — Collection — Song")
    print('-' * 100)
    for ratio, fpath in results[:top_k]:
        composer, collection, song = corpus_info(fpath.name)
        meta = f"{composer} — {collection} — {song}" if composer else "(not found in corpus)"
        print(f"{ratio:>6.4f}  {fpath.name:<28}  {meta}")

    top_score = results[0][0] if results else 0.0
    if top_score > 0.9:
        print("\n WARNING: Very high interval similarity ({:.0%}) — possible memorization/overfitting!".format(top_score))
    elif top_score > 0.7:
        print("\n High interval similarity ({:.0%}) — review top matches.".format(top_score))
    else:
        print(f"\nMax interval similarity: {top_score:.4f} — output appears novel.")

check_similarity_intervals(filename)


Scanning 21295 training files (interval-based, all keys)...
Generated meter: simple, 4 beats — filtering by matching meter first.
Skipped 13815 non-matching-meter files. Comparing 7480 candidates.

Top 10 most similar training files to 'Romantic_Schubert_Franz_Art_Song.abc' (interval-based):
   Sim  File                          Composer — Collection — Song
----------------------------------------------------------------------------------------------------
0.1668  lc6478017_Gb.abc              Bridge, Frank —   — The Last Invocation, H.136
0.1636  lc6478017_G.abc               Bridge, Frank —   — The Last Invocation, H.136
0.1606  lc6620711_Gb.abc              Gray, Louisa —   — Unforgotten
0.1606  lc6620711_G.abc               Gray, Louisa —   — Unforgotten
0.1606  lc6620711_Db.abc              Gray, Louisa —   — Unforgotten
0.1606  lc6620711_D.abc               Gray, Louisa —   — Unforgotten
0.1606  lc6620711_Ab.abc              Gray, Louisa —   — Unforgotten
0.1606  lc6620711_A.abc 

In [17]:
# periods = {
#     "Baroque": [
#         # "Bach, Johann Sebastian",
#         # "Corelli, Arcangelo",
#         # "Handel, George Frideric",
#         # "Scarlatti, Domenico",
#         "Vivaldi, Antonio"
#     ],
#     "Classical": [
#         "Beethoven, Ludwig van",
#         "Haydn, Joseph",
#         "Mozart, Wolfgang Amadeus",
#         "Paradis, Maria Theresia von",
#         "Reichardt, Louise",
#         "Saint-Georges, Joseph Bologne",
#         "Schroter, Corona"
#     ],
#     "Romantic": [
#         "Bartok, Bela",
#         "Berlioz, Hector",
#         "Bizet, Georges",
#         "Boulanger, Lili",
#         "Boulton, Harold",
#         "Brahms, Johannes",
#         "Burgmuller, Friedrich",
#         "Butterworth, George",
#         "Chaminade, Cecile",
#         "Chausson, Ernest",
#         "Chopin, Frederic",
#         "Cornelius, Peter",
#         "Debussy, Claude",
#         "Dvorak, Antonin",
#         "Faisst, Clara",
#         "Faure, Gabriel",
#         "Franz, Robert",
#         "Gonzaga, Chiquinha",
#         "Grandval, Clemence de",
#         "Grieg, Edvard",
#         "Hensel, Fanny",
#         "Holmes, Augusta Mary Anne",
#         "Jaell, Marie",
#         "Kinkel, Johanna",
#         "Kralik, Mathilde",
#         "Lang, Josephine",
#         "Lehmann, Liza",
#         "Liszt, Franz",
#         "Mayer, Emilie",
#         "Medtner, Nikolay",
#         "Mendelssohn, Felix",
#         "Munktell, Helena",
#         "Parratt, Walter",
#         "Prokofiev, Sergey",
#         "Rachmaninoff, Sergei",
#         "Ravel, Maurice",
#         "Saint-Saens, Camille",
#         "Satie, Erik",
#         "Schubert, Franz",
#         "Schumann, Clara",
#         "Schumann, Robert",
#         "Scriabin, Aleksandr",
#         "Shostakovich, Dmitry",
#         "Sibelius, Jean",
#         "Smetana, Bedrich",
#         "Tchaikovsky, Pyotr"
#     ]
# }


# def sanitize(name: str) -> str:
#     """Convert to safe filename (remove spaces, commas, etc.)."""
#     return re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_')

# instrumentations = [
#                     # "Keyboard",
#                     "Orchestral", 
#                     "Choral", 
#                     "Chamber", 
#                     "Art Song"
#                     ]

# for period, composers in periods.items():
#     for composer in composers:
#         for instrumentation in instrumentations:
#             print(f"Period: {period}, Composer: {composer}, Instrumentation: {instrumentation}")
            

#             try:
#                 result = inference_patch(period, composer, instrumentation)

#                 abc_lines = result.splitlines()
#                 abc_lines = [line + '\n' for line in abc_lines if line.strip()]  # Add newlines and remove empty lines

#                 abc_lines = rest_unreduce(abc_lines)

#                 # Construct unique file name
#                 filename = f"{sanitize(period)}_{sanitize(composer)}_{sanitize(instrumentation)}.abc"
                
#                 with open(filename, "w", encoding="utf-8") as f:
#                     f.writelines(abc_lines)
#                 !python abc2xml.py -o . {filename}
#             except Exception as e:
#                 print(f"Some error {e} has occured, skipping{filename}")
#                 continue


In [18]:

# def sanitize(name: str) -> str:
#     """Convert to safe filename (remove spaces, commas, etc.)."""
#     return re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_')

# for i in range(10):
#     try:
#         period, composer, instrumentation = 'Classical', 'Beethoven, Ludwig van', 'Keyboard'
#         default_note, tempo, meter, key = "1/8", "1/4=60", "3/4", "C"
#         # score = "1 { ( 2 4 ) | ( 3 5 ) }"
#         score = "{ ( 1 2 ) | ( 3 4 ) }"
#         voices = ["""V:1 treble nm="piano"\n""",
#                     """V:2 treble\n""",
#                     """V:3 bass nm="piano"\n""",
#                     """V:4 bass\n"""]
#         first_tune = """[r:0/80][V:1]"^Andante con moto"!p! g4{a} (gf/e/)|[V:3](C/G/E/G/) (C/G/E/G/) (C/G/E/G/)|\n"""
#         result = inference_patch(period, composer, instrumentation, [f"%%score {score} \n", f"L:{default_note}\n", f"Q:{tempo}\n", f"M:{meter}\n", f"K:{key}\n"]+ voices, [first_tune])

#         abc_lines = result.splitlines()
#         abc_lines = [line + '\n' for line in abc_lines if line.strip()]  # Add newlines and remove empty lines

#         abc_lines = rest_unreduce(abc_lines)

#         # Construct unique file name
#         filename = f"fine_tuned_{sanitize(period)}_{sanitize(composer)}_{sanitize(instrumentation)}_{i}.abc"

#         with open(filename, "w", encoding="utf-8") as f:
#             f.writelines(abc_lines)
#         !python abc2xml.py -o . {filename}
#     except Exception as e:
#         print(f"Exception caught {e}, skipping {sanitize(period)}_{sanitize(composer)}_{sanitize(instrumentation)}")

In [19]:
# try:
#     period, composer, instrumentation = 'Classical', 'Beethoven, Ludwig van', 'Art Song'
#     default_note, tempo, meter, key = "1/8", "1/4=80", "3/4", "D"
#     score = "1 { ( 2 4 ) | ( 3 5 ) }"
#     # score = "[ 1 2 3 4 ]"
#     # score = ""
#     # print([f"%%score {score} \n", f"L:{default_note}\n", f"Q:{tempo}\n", f"M:{meter}\n", f"K:{key}\n"])
#     result = inference_patch(period, composer, instrumentation, [f"%%score {score} \n", 
#     f"L:{default_note}\n", 
#     f"Q:{tempo}\n", 
#     f"M:{meter}\n", 
#     f"K:{key}\n"])

#     abc_lines = result.splitlines()
#     abc_lines = [line + '\n' for line in abc_lines if line.strip()]  # Add newlines and remove empty lines

#     abc_lines = rest_unreduce(abc_lines)

#     # Construct unique file name
#     filename = f"{sanitize(period)}_{sanitize(composer)}_{sanitize(instrumentation)}_{i}.abc"

#     with open(filename, "w", encoding="utf-8") as f:
#         f.writelines(abc_lines)
#     !python abc2xml.py -o . {filename}
# except Exception as e:
#     print(f"Exception caught {e}, skipping {sanitize(period)}_{sanitize(composer)}_{sanitize(instrumentation)}")

In [20]:
# try:
#     period, composer, instrumentation = 'Classical', 'Haydn, Joseph', 'Chamber'
#     default_note, tempo, meter, key = "1/8", "1/4=60", "3/4", "C"
#     score = "1 { ( 2 4 ) | ( 3 5 ) }"
#     # score = "{ ( 1 2 ) | ( 3 4 ) }"
#     result = inference_patch(period, composer, instrumentation, [f"%%score {score} \n", f"L:{default_note}\n", f"Q:{tempo}\n", f"M:{meter}\n", f"K:{key}\n"])

#     abc_lines = result.splitlines()
#     abc_lines = [line + '\n' for line in abc_lines if line.strip()]  # Add newlines and remove empty lines

#     abc_lines = rest_unreduce(abc_lines)

#     # Construct unique file name
#     filename = f"{sanitize(period)}_{sanitize(composer)}_{sanitize(instrumentation)}_{i}.abc"

#     with open(filename, "w", encoding="utf-8") as f:
#         f.writelines(abc_lines)
#     !python abc2xml.py -o . {filename}
# except Exception as e:
#     print(f"Exception caught {e}, skipping {sanitize(period)}_{sanitize(composer)}_{sanitize(instrumentation)}")

In [21]:
# try:
#     period, composer, instrumentation = 'Classical', 'Beethoven, Ludwig van', 'Keyboard'
#     default_note, tempo, meter, key = "1/8", "1/4=60", "3/4", "C"
#     # score = "1 { ( 2 4 ) | ( 3 5 ) }"
#     score = "{ ( 1 2 ) | ( 3 4 ) }"
#     result = inference_patch(period, composer, instrumentation, [f"%%score {score} \n", f"L:{default_note}\n", f"Q:{tempo}\n", f"M:{meter}\n", f"K:{key}\n"])

#     abc_lines = result.splitlines()
#     abc_lines = [line + '\n' for line in abc_lines if line.strip()]  # Add newlines and remove empty lines

#     abc_lines = rest_unreduce(abc_lines)

#     # Construct unique file name
#     filename = f"{sanitize(period)}_{sanitize(composer)}_{sanitize(instrumentation)}_{i}.abc"

#     with open(filename, "w", encoding="utf-8") as f:
#         f.writelines(abc_lines)
#     !python abc2xml.py -o . {filename}
# except Exception as e:
#     print(f"Exception caught {e}, skipping {sanitize(period)}_{sanitize(composer)}_{sanitize(instrumentation)}")

In [22]:
# === Attention-weight inspection ===
# How much attention do non-motif queries actually pay to motif key positions?
# Compare bias-ON (the value the model was trained with) vs bias-OFF (what
# generate() currently uses at inference) so you can see (a) whether the bias
# meaningfully shifts attention probabilities and (b) the order of magnitude.
# Per-layer columns:
#   share->motif   fraction of attention from non-motif queries to motif keys
#   avg/motif key  share / (# motif keys)  -> per-key probability mass
#   avg/non-motif  same per-key probability mass for the rest of the sequence
#   log10 ratio    positive => motif keys get systematically more attention each
compare_bias_on_off(model, patchilizer, period, composer, instrumentation, [motif])

=== With training bias (MOTIF_ATTENTION_BIAS = 2.00) ===


RuntimeError: The size of tensor a (15) must match the size of tensor b (14) at non-singleton dimension 3